# Verifikasi perbaikan adapter dan SENA v0.5

Notebook ini benar-benar menjalankan pemeriksaan kontrak dan inferensi lokal CPU. Target dan cutoff 0,95 tidak diubah. Replay field menggunakan prediksi mentah dan acuan manusia lama yang tetap disimpan. Inferensi berikutnya memakai bobot SENA baru yang hash-nya diperiksa dan bobot NARA kandidat yang sama.

Lima puluh contoh lama telah dipakai untuk memperbaiki adapter sehingga merupakan data regresi pengembangan. Dua puluh empat narasi tambahan memiliki acuan buatan asisten dan hanya dipakai sekali setelah pemilihan checkpoint. Kedua hasil tidak memberi persetujuan mutu manusia secara otomatis. Encoder tidak dilatih ulang di notebook ini. Satu bias head NON_EVENT dipilih dari data pengembangan sebelum pengujian; ini fitting parameter pengembangan dan bukan bukti kalibrasi probabilitas atau validasi independen.

In [1]:
from pathlib import Path
import os,sys,subprocess,json
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
ASSETS=Path(os.environ['SLR09_ASSET_DIR'])
BASE=Path(os.environ['SLR09_TRAIN_OUTPUT']).parent
PREVIOUS=ROOT/'evidence/quality_fix_v05/previous_50'
SENA_DIR=Path(os.environ.get('SLR09_QUALITY_SENA_DIR',str(BASE/'prior_adjustment_20261019_cpu/best_model')))
OUTPUT=BASE/'quality_diagnostics_v05'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Pemeriksaan gagal: '+str(p.returncode))
print('Inference device: CPU. Target dan acuan lama tetap.')

Inference device: CPU. Target dan acuan lama tetap.


In [2]:
execute([sys.executable,str(ROOT/'code/quality_fix/fit_sena_prior.py'),
 '--source',str(BASE/'training_run_20261019_cpu_focus'),
 '--output',str(BASE/'prior_adjustment_20261019_cpu')])
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_quality_candidate.py'),
 '--run',str(BASE/'prior_adjustment_20261019_cpu'),'--exports',str(BASE/'exports')])

{
  "bias_added_to_non_event_head": 2.1,
  "development": {
    "accuracy": 0.8904109589041096,
    "macro_f1": 0.8988168570955457,
    "per_class": {
      "FLOOD": {
        "support": 32,
        "precision": 0.9310344827586207,
        "recall": 0.84375,
        "f1": 0.8852459016393442
      },
      "EARTHQUAKE": {
        "support": 24,
        "precision": 1.0,
        "recall": 0.875,
        "f1": 0.9333333333333333
      },
      "FIRE": {
        "support": 35,
        "precision": 1.0,
        "recall": 0.8857142857142857,
        "f1": 0.9393939393939394
      },
      "ACCIDENT": {
        "support": 24,
        "precision": 0.9583333333333334,
        "recall": 0.9583333333333334,
        "f1": 0.9583333333333334
      },
      "NON_EVENT": {
        "support": 31,
        "precision": 0.6829268292682927,
        "recall": 0.9032258064516129,
        "f1": 0.7777777777777778
      }
    },
    "predeclared_selection_metric": 1.003584229390681,
    "targets_met": true
  

In [3]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),'--evidence-dir',str(BASE/'contract_tests_v05')])

test_explicit_breathing_condition_is_kept (test_candidate_runtime.ActiveNormalizationTests.test_explicit_breathing_condition_is_kept) ... ok
test_literal_impact_still_maps_to_impact (test_candidate_runtime.ActiveNormalizationTests.test_literal_impact_still_maps_to_impact) ... ok
test_negated_breathing_condition_is_not_positive_victim (test_candidate_runtime.ActiveNormalizationTests.test_negated_breathing_condition_is_not_positive_victim) ... ok
test_unconscious_condition_is_not_misread_as_negation (test_candidate_runtime.ActiveNormalizationTests.test_unconscious_condition_is_not_misread_as_negation) ... ok
test_candidate_directory_has_explicit_identity_and_hashes (test_candidate_runtime.CandidateAssetTests.test_candidate_directory_has_explicit_identity_and_hashes) ... ok
test_config_labels_must_match_even_if_hash_was_updated (test_candidate_runtime.CandidateAssetTests.test_config_labels_must_match_even_if_hash_was_updated) ... ok
test_different_weights_cannot_be_accepted_as_candidate (

In [4]:
execute([sys.executable,str(ROOT/'code/quality_fix/replay_fields.py'),
 '--source',str(PREVIOUS),'--output',str(BASE/'field_replay_final_v05')])

{
  "scope": "SEEN_PILOT_DEVELOPMENT_REPLAY_NOT_ACCEPTANCE",
  "adapter_version": "silar-adapter-0.5.0",
  "nonempty_exact": 0.9162303664921466,
  "empty_spurious_rate": 0.012578616352201259,
  "checks": {
    "sena_macro_f1": false,
    "non_event_recall": false,
    "all_event_class_recalls": true,
    "nara_macro_f1_all_14_non_o": true,
    "nara_exact_span_f1": true,
    "fields_nonempty_exact": true,
    "empty_field_spurious_rate": true,
    "adapter_fusion_contract": true,
    "simulation_vs_real_exercise_contrasts": false
  }
}


In [5]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_quality_diagnostics.py'),
 '--previous',str(PREVIOUS),'--challenge',str(ROOT/'data/quality_fix_fresh_challenge_v0.5.json'),
 '--sena-dir',str(SENA_DIR),'--nara-archive',str(ASSETS/'SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--output',str(OUTPUT)])

Seen pilot inference 1 / 50
Seen pilot inference 10 / 50
Seen pilot inference 20 / 50
Seen pilot inference 30 / 50
Seen pilot inference 40 / 50
Seen pilot inference 50 / 50
{
  "seen_pilot": {
    "sena_macro_f1": 1.0,
    "non_event_recall": 1.0,
    "fields_nonempty_exact": 0.9162303664921466,
    "empty_spurious_rate": 0.012578616352201259,
    "checks": {
      "sena_macro_f1": true,
      "non_event_recall": true,
      "all_event_class_recalls": true,
      "nara_macro_f1_all_14_non_o": true,
      "nara_exact_span_f1": true,
      "fields_nonempty_exact": true,
      "empty_field_spurious_rate": true,
      "adapter_fusion_contract": true,
      "simulation_vs_real_exercise_contrasts": true
    }
  },
  "fresh_challenge": {
    "accuracy": 0.8333333333333334,
    "macro_f1": 0.8006593406593406,
    "non_event_recall": 0.9166666666666666
  },
  "quality_acceptance_approved": false
}


In [6]:
report=json.loads((OUTPUT/'evaluation.json').read_text())
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['quality_acceptance_approved'] and not state['card_done']
print('Hasil nyata tersimpan. Status mutu memerlukan penilaian terpisah.')
print(json.dumps({'checks':report['checks'],'quality_acceptance_approved':False,'card_done':False},indent=2))

Hasil nyata tersimpan. Status mutu memerlukan penilaian terpisah.
{
  "checks": {
    "sena_macro_f1": true,
    "non_event_recall": true,
    "all_event_class_recalls": true,
    "nara_macro_f1_all_14_non_o": true,
    "nara_exact_span_f1": true,
    "fields_nonempty_exact": true,
    "empty_field_spurious_rate": true,
    "adapter_fusion_contract": true,
    "simulation_vs_real_exercise_contrasts": true
  },
  "quality_acceptance_approved": false,
  "card_done": false
}
